# 1. Setup

In [ ]:
!git clone --branch update-prompts --single-branch https://github.com/longkvy/mura-finance.git

fatal: destination path 'mura-finance' already exists and is not an empty directory.


In [ ]:
%cd /content/mura-finance/

/content/mura-finance


In [1]:
from pathlib import Path
ROOT = Path(".")

# ROOT = Path("/content/mura-finance")
# !pip install -q -r requirements.txt
# !pip install accelerate sentencepiece attrdict tqdm scikit-learn
# print("Requirements installed. Project root:", ROOT)

import numpy as np

def sentiment_str_to_numeric(s: str) -> float:
    """Map pipeline sentiment to dataset labels: Positive->1, Negative->-1, Neutral->0."""
    if s is None:
        return np.nan
    s = (s or "").strip().lower()
    if s == "positive":
        return 1.0
    if s == "negative":
        return -1.0
    if s == "neutral":
        return 0.0
    return np.nan

# 2. Ollama Solution

In [ ]:
!sudo apt update
!sudo apt install -y pciutils zstd systemd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
66 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as re

In [ ]:
import threading
import subprocess
import time

def run_ollama_serve():
  subprocess.Popen(["ollama", "serve"])

thread = threading.Thread(target=run_ollama_serve)
thread.start()
time.sleep(5)

In [1]:
# !ollama pull qwen3
!ollama pull gemma3:1b
# !ollama pull phi4-mini-reasoning
# !ollama pull qwen3:14b
#!ollama pull gemma3:12b


]11;?\pulling manifest ⠙ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 7cd4618c1faf: 100% ▕██████████████████▏ 815 MB                         
pulling e0a42594d802: 100% ▕██████████████████▏  358 B                         
pulling dd084c7d92a3: 100% ▕██████████████████▏ 8.4 KB                         
pulling 3116c5225075: 100% ▕██████████████████▏   77 B                         
pulling 120007c81bf8: 100% ▕██████████████████▏  492 B                         
verifying sha256 digest 
writing manifest 
success 


In [2]:
import sys
import os
from pathlib import Path

import pandas as pd

# ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
# if not (ROOT / "src").exists():
#     ROOT = Path("/content/mura-finance")
# sys.path.insert(0, str(ROOT))
# os.chdir(ROOT)

test_model = "gemma3:1b"

os.environ.setdefault("OLLAMA_HOST", "http://localhost:11434")
os.environ.setdefault("OLLAMA_MODEL", test_model)

'gemma3:1b'

In [3]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path().resolve().parent))

from src.utils.data_loader import load_all_dataframes
from src.evaluation.metrics import compute_classification_metrics, confusion_matrix_dict

N_SAMPLES = 10
# base = Path("/content/mura-finance")
base = Path().resolve().parent

data = load_all_dataframes(base)

dev = data["dev"]
sample = dev.head(N_SAMPLES).copy()

print(f"Loaded {len(sample)} samples. Columns: text, ticker, true_sentiment")
sample[["text", "ticker", "true_sentiment"]].head(3)

Loaded dev split: 1829 rows from /Users/longnguyen/Desktop/uoa-group1-c6/data/dev.csv
Loaded test split: 462 rows from /Users/longnguyen/Desktop/uoa-group1-c6/data/test.csv
Loaded 10 samples. Columns: text, ticker, true_sentiment


,text,ticker,true_sentiment
0,Economists at Credit Suisse are now neutral o...,EURCHF,Negative
1,New low for the EURUSD as it remains the curre...,EURCHF,Negative
2,EUR/CHF vaults parity for the first time since...,EURCHF,Neutral


In [4]:
from src.pipeline.llm_client import LLMClient
from src.pipeline.orchestrator import ReasoningPipeline

llm = LLMClient(max_tokens=1024, model=test_model)
pipeline = ReasoningPipeline(llm_client=llm)
print("LLMClient and ReasoningPipeline ready.")

LLMClient and ReasoningPipeline ready.


In [5]:
y_true_5hop = sample["true_sentiment"].apply(sentiment_str_to_numeric).values.astype(float)
y_pred_5hop = np.full(len(sample), np.nan, dtype=float)

if pipeline is not None:
    for idx in range(len(sample)):
        row = sample.iloc[idx]
        # text = row.get("text") or row.get("title") or ""
        text = row.get("title")
        ticker = row.get("ticker")
        if pd.isna(ticker):
            ticker = None
        else:
            ticker = str(ticker).strip() or None
        try:
            context = pipeline.run(text, ticker=ticker)
            print(context)
            sent = context.sentiment
            y_pred_5hop[idx] = sentiment_str_to_numeric(sent)
        except Exception as e:
            print(f"Sample {idx}: {e}")
            y_pred_5hop[idx] = np.nan
    print(
        f"Completed. Valid predictions: {np.sum(np.isfinite(y_pred_5hop))} / {len(sample)}"
    )
else:
    print("Pipeline not initialized; skipping run.")

ReasoningContext(text='EURCHF Bias would be for stronger Franc but waiting for clearer SNB monetary policy stance – Credit Suisse', ticker='EURCHF', fx_insight='Okay, here’s a breakdown of the FX signal for EURCHF and YEN (YEN) based on the provided information, focusing solely on the headline and its implications:\n\n**EURCHF**\n\n*   **Currency Affected:** EURCHF\n*   **Direction of Pressure:** Downward\n*   **Rationale:** The headline suggests a stronger Franc, but the wait for SNB’s policy stance indicates uncertainty.  This suggests a potential weakening trend.\n\n**YEN**\n\n*   **Currency Affected:** YEN\n*   **Direction of Pressure:**  No clear pressure detected.\n*   **Rationale:** The headline focuses on SNB’s policy, not a specific direction for the YEN.\n\n\n---\n\n**Important Disclaimer:** *This analysis is based solely on the provided headline and limited context.  FX markets are complex and influenced by numerous factors. This is a preliminary assessment and should not be

In [7]:
metrics_5hop = compute_classification_metrics(y_true_5hop, y_pred_5hop)

if "error" in metrics_5hop:
    print("Error:", metrics_5hop["error"])
else:
    print("5-Hop Pipeline — Metrics (same 100 samples)")
    print("-" * 50)
    print(f"  n (valid pairs): {metrics_5hop['n']}")
    print(f"  Accuracy:        {metrics_5hop['accuracy']:.4f}")
    print(f"  F1 (macro):     {metrics_5hop['f1_macro']:.4f}")
    print(f"  Precision (macro): {metrics_5hop['precision_macro']:.4f}")
    print(f"  Recall (macro):   {metrics_5hop['recall_macro']:.4f}")

# Confusion matrix
if "error" not in metrics_5hop:
    cm_5hop = confusion_matrix_dict(y_true_5hop, y_pred_5hop)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print(
        pd.DataFrame(
            cm_5hop["matrix"], index=cm_5hop["labels"], columns=cm_5hop["labels"]
        )
    )

5-Hop Pipeline — Metrics (same 100 samples)
--------------------------------------------------
  n (valid pairs): 10
  Accuracy:        0.6000
  F1 (macro):     0.2500
  Precision (macro): 0.2000
  Recall (macro):   0.3333

Confusion matrix (rows=true, cols=pred):
          Negative  Neutral  Positive
Negative         0        0         2
Neutral          0        0         2
Positive         0        0         6


In [9]:
sample

,published_at,ticker,true_sentiment,title,author,url,source,text,finbert_sentiment,finbert_sent_score,external_context_1,external_context_2,external_context_3
0,2023-01-18 15:24:00,EURCHF,Negative,EURCHF Bias would be for stronger Franc but wa...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-bias-wou...,FX Street,Economists at Credit Suisse are now neutral o...,Neutral,0.08,"Financial Sector Update for 01/18/2023: PNC, B...","Financial Sector Update for 01/18/2023: PNC, B...","Financial Sector Update for 01/18/2023: PNC, B..."
1,2023-01-20 04:59:00,EURCHF,Negative,New lows for the EURUSD. EURCHF down as well a...,Greg Michalowski,https://www.forexlive.com/technical-analysis/n...,Forex Live,New low for the EURUSD as it remains the curre...,Negative,-0.93,European Stocks Climb Early in New Year: An ET...,Vanguard Real Estate ETF Experiences Big Outfl...,Vanguard Real Estate ETF Experiences Big Outfl...
2,2023-01-12 11:40:00,EURCHF,Neutral,Does a jump in EURCHF point to a break above 1...,FXStreet Insights Team,https://www.fxstreet.com/news/does-a-jump-in-e...,FX Street,EUR/CHF vaults parity for the first time since...,Neutral,0.37,"Down -29.05% in 4 Weeks, Here's Why You Should...",CEE MARKETS-FX drifts before U.S. inflation da...,Bear Market Watch: What Will Mean a Bull Marke...
3,2023-01-13 19:48:00,EURCHF,Neutral,USDCHF stalls its run higher at the 200 bar MA...,Greg Michalowski,https://www.forexlive.com/technical-analysis/u...,Forex Live,The USDCHF is sharply higher on the day helped...,Positive,0.77,CEE MARKETS-Forint climbs back near 5-month hi...,BBCA Breaks Above 200-Day Moving Average - Bul...,EWC Makes Bullish Cross Above Critical Moving ...
4,2023-01-12 07:47:00,EURCHF,Positive,Euro to benefit from the ECBs pronounced hawki...,FXStreet Insights Team,https://www.fxstreet.com/news/euro-to-benefit-...,FX Street,The Euro was able to appreciate particularly s...,Positive,0.85,CEE MARKETS-FX drifts before U.S. inflation da...,European shares rise as investors await U.S. i...,MOVES-Rothschild & Co appoint Horn as capital ...
5,2023-01-31 09:58:00,EURCHF,Positive,EURCHF to remain well supported amid widening ...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-remai...,FX Street,Widening policy rate differential withECBsugge...,Positive,0.87,ETF Prime: Disruptive Tech and International S...,GLOBAL MARKETS-World stocks waver as investors...,5 ETFs That Gained Investors' Love Last Week\n...
6,2023-01-12 15:32:00,EURCHF,Positive,EURCHF could extend its advance back to levels...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-could-ex...,FX Street,EUR/CHF climbs back above parity. Economists a...,Positive,0.64,CEE MARKETS-FX drifts before U.S. inflation da...,Bullish Two Hundred Day Moving Average Cross -...,"SPSM, AMN, FN, SPSC: ETF Inflow Alert\n\nLooki..."
7,2023-01-13 17:05:00,EURCHF,Positive,EURCHF Room for the Euro to extend the move hi...,Matías Salord,https://www.fxstreet.com/news/eur-chf-room-for...,FX Street,Analysts at MUFG Bank have a bullish outlook f...,Positive,0.80,CEE MARKETS-Forint climbs back near 5-month hi...,Euronav NV (EURN) Soars 8.9%: Is Further Upsid...,Check Out OEUR for Exposure to Rally in Europe...
8,2023-01-13 11:37:00,EURCHF,Positive,EURCHF to head higher towards 10130 and projec...,FXStreet Insights Team,https://www.fxstreet.com/news/eur-chf-to-head-...,FX Street,EUR/CHF has broken out above the sideways rang...,Positive,0.83,CEE MARKETS-Forint climbs back near 5-month hi...,"SOXS, EGPT: Big ETF Inflows\n\nComparing units...","SOXS, EGPT: Big ETF Inflows\n\nComparing units..."
9,2023-01-27 16:07:00,EURCHF,Positive,EURCHF Still room to rise toward the 10500 are...,Matías Salord,https://www.fxstreet.com/news/eur-chf-still-ro...,FX Street,The EUR/CHF pair has climbed back above parity...,Positive,0.83,"XLF, YANG: Big ETF Outflows\n\nLooking at unit...","XLF, YANG: Big ETF Outflows\n\nLooking at unit...","XLF, YANG: Big ETF Outflows\n\nLooking at unit..."


# 3. Flan T5 XXL

In [ ]:
from src.pipeline import ReasoningPipeline
from src.pipeline.llm_client import LLMClient

llm = LLMClient(
    provider="flan_t5",
    model="google/flan-t5-xxl",
    max_tokens=2048,
    temperature=0.0,
    device=None,
)

pipeline = ReasoningPipeline(llm_client=llm)
print(f"Provider: {llm.provider}")
print("Model: google/flan-t5-xxl (loads on first generate)")

Provider: flan_t5
Model: google/flan-t5-xxl (loads on first generate)


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path().resolve().parent))

from src.utils.data_loader import load_all_dataframes
from src.evaluation.metrics import compute_classification_metrics, confusion_matrix_dict

N_SAMPLES = 100
base = Path("/content/mura-finance")
data = load_all_dataframes(base)

sa = data["single_article"]
sample = sa.head(N_SAMPLES).copy()

print(f"Loaded {len(sample)} samples. Columns: text, ticker, true_sentiment")
sample[["text", "ticker", "true_sentiment"]].head(3)

Loaded ground truth: 2291 rows
Loaded single article predictions: 2291 rows
Loaded all-day articles: 293 rows
Loaded 100 samples. Columns: text, ticker, true_sentiment


,text,ticker,true_sentiment
0,The Euro was able to appreciate particularly s...,EURCHF,1
1,EUR/CHF yesterday broke above 1.00. Economists...,EURCHF,1
2,EUR/CHF vaults parity for the first time since...,EURCHF,0


In [ ]:
y_true_5hop = sample["true_sentiment"].values.astype(float)
y_pred_5hop = np.full(len(sample), np.nan, dtype=float)

if pipeline is not None:
    for idx in range(len(sample)):
        row = sample.iloc[idx]
        # text = row.get("text") or row.get("title") or ""
        text = row.get("title")
        ticker = row.get("ticker")
        if pd.isna(ticker):
            ticker = None
        else:
            ticker = str(ticker).strip() or None
        try:
            context = pipeline.run(text, ticker=ticker)
            print(context)
            print("*************")
            sent = context.sentiment
            y_pred_5hop[idx] = sentiment_str_to_numeric(sent)
        except Exception as e:
            print(f"Sample {idx}: {e}")
            y_pred_5hop[idx] = np.nan
    print(
        f"Completed. Valid predictions: {np.sum(np.isfinite(y_pred_5hop))} / {len(sample)}"
    )
else:
    print("Pipeline not initialized; skipping run.")

Loading weights:   0%|          | 0/559 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ReasoningContext(text='Euro to benefit from the ECBs pronounced hawkish determination – Commerzbank', ticker='EURCHF', fx_insight='Upward pressure', base_sentiment='Positive', quote_sentiment='Neutral', sentiment='Positive', hop_results={'fx_insight': {'fx_insight': 'Upward pressure', 'raw_response': 'Upward pressure'}, 'base_currency_sentiment': {'base_sentiment': 'Positive', 'raw_response': 'Positive'}, 'quote_currency_sentiment': {'quote_sentiment': 'Neutral', 'raw_response': 'Neutral'}, 'final_classification': {'sentiment': 'Positive', 'raw_response': 'Positive'}}, raw_responses={'fx_insight': 'Upward pressure', 'base_currency_sentiment': 'Positive', 'quote_currency_sentiment': 'Neutral', 'final_classification': 'Positive'})
*************
ReasoningContext(text='EURCHF Trend higher may remain in place – ING', ticker='EURCHF', fx_insight='Upward pressure', base_sentiment='Positive', quote_sentiment='Positive', sentiment='Positive', hop_results={'fx_insight': {'fx_insight': 'Upward pr

In [ ]:
metrics_5hop = compute_classification_metrics(y_true_5hop, y_pred_5hop)

if "error" in metrics_5hop:
    print("Error:", metrics_5hop["error"])
else:
    print("5-Hop Pipeline — Metrics (same 100 samples)")
    print("-" * 50)
    print(f"  n (valid pairs): {metrics_5hop['n']}")
    print(f"  Accuracy:        {metrics_5hop['accuracy']:.4f}")
    print(f"  F1 (macro):     {metrics_5hop['f1_macro']:.4f}")
    print(f"  Precision (macro): {metrics_5hop['precision_macro']:.4f}")
    print(f"  Recall (macro):   {metrics_5hop['recall_macro']:.4f}")

# Confusion matrix
if "error" not in metrics_5hop:
    cm_5hop = confusion_matrix_dict(y_true_5hop, y_pred_5hop)
    print("\nConfusion matrix (rows=true, cols=pred):")
    print(
        pd.DataFrame(
            cm_5hop["matrix"], index=cm_5hop["labels"], columns=cm_5hop["labels"]
        )
    )

5-Hop Pipeline — Metrics (same 100 samples)
--------------------------------------------------
  n (valid pairs): 100
  Accuracy:        0.7200
  F1 (macro):     0.7161
  Precision (macro): 0.7374
  Recall (macro):   0.7188

Confusion matrix (rows=true, cols=pred):
          Negative  Neutral  Positive
Negative        20       13         1
Neutral          3       21         8
Positive         0        3        31
